# 04 - Baseline Model Validation

このノートブックでは、複数のベースラインモデルを比較し、最良のアプローチを選定します。

## Contents
1. データ読み込み
2. K-Fold交差検証の設定
3. モデル比較（LR, LightGBM, XGBoost, CatBoost）
4. パフォーマンス分析
5. 最良モデルの選択

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import lightgbm as lgb
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Paths
TRAIN_PATH = 'data/processed/train_engineered_selected.csv'
ANALYSIS_DIR = Path('data/analysis')

print('Imports successful!')

## 1. データ読み込み

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)

target_col = 'Irrigation_Need'
y = train_df[target_col].values
X = train_df.drop([target_col], axis=1).values
feature_names = train_df.drop([target_col], axis=1).columns.tolist()

print(f'Data shape: {X.shape}')
print(f'Target distribution: {np.bincount(y)}')
print(f'Features: {len(feature_names)}')

## 2. K-Fold 交差検証

In [ ]:
# Setup K-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for results
cv_results = {}

print(f'Starting {n_splits}-Fold Cross-Validation...\n')

## 3. Logistic Regression Baseline

In [ ]:
lr_scores = {'AUC': [], 'F1': [], 'Precision': [], 'Recall': []}

print('=' * 50)
print('Logistic Regression')
print('=' * 50)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_valid)[:, 1]
    y_pred = model.predict(X_valid)
    
    auc = roc_auc_score(y_valid, y_pred_proba)
    f1 = f1_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred)
    rec = recall_score(y_valid, y_pred)
    
    lr_scores['AUC'].append(auc)
    lr_scores['F1'].append(f1)
    lr_scores['Precision'].append(prec)
    lr_scores['Recall'].append(rec)
    
    print(f'Fold {fold+1}: AUC={auc:.4f}, F1={f1:.4f}, Prec={prec:.4f}, Rec={rec:.4f}')

print(f'\nAvg AUC: {np.mean(lr_scores["AUC"]):.4f} (+/- {np.std(lr_scores["AUC"]):.4f})')
cv_results['LogisticRegression'] = lr_scores

## 4. LightGBM

In [ ]:
lgbm_scores = {'AUC': [], 'F1': [], 'Precision': [], 'Recall': []}

print('\n' + '=' * 50)
print('LightGBM')
print('=' * 50)

lgbm_params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'random_state': 42,
}

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
    
    model = lgb.train(
        lgbm_params,
        train_data,
        num_boost_round=500,
        valid_sets=[valid_data],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    y_pred_proba = model.predict(X_valid)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    auc = roc_auc_score(y_valid, y_pred_proba)
    f1 = f1_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred)
    rec = recall_score(y_valid, y_pred)
    
    lgbm_scores['AUC'].append(auc)
    lgbm_scores['F1'].append(f1)
    lgbm_scores['Precision'].append(prec)
    lgbm_scores['Recall'].append(rec)
    
    print(f'Fold {fold+1}: AUC={auc:.4f}, F1={f1:.4f}, Prec={prec:.4f}, Rec={rec:.4f}')

print(f'\nAvg AUC: {np.mean(lgbm_scores["AUC"]):.4f} (+/- {np.std(lgbm_scores["AUC"]):.4f})')
cv_results['LightGBM'] = lgbm_scores

## 5. モデル比較

In [ ]:
# Comparison table
comparison = []
for model_name, scores in cv_results.items():
    comparison.append({
        'Model': model_name,
        'AUC': f"{np.mean(scores['AUC']):.4f} (+/- {np.std(scores['AUC']):.4f})",
        'F1': f"{np.mean(scores['F1']):.4f} (+/- {np.std(scores['F1']):.4f})",
        'Precision': f"{np.mean(scores['Precision']):.4f} (+/- {np.std(scores['Precision']):.4f})",
        'Recall': f"{np.mean(scores['Recall']):.4f} (+/- {np.std(scores['Recall']):.4f})",
    })

comparison_df = pd.DataFrame(comparison)
print('\n' + '=' * 80)
print('MODEL COMPARISON')
print('=' * 80)
print(comparison_df.to_string(index=False))

# Save results
with open(ANALYSIS_DIR / 'baseline_cv_results.json', 'w') as f:
    json.dump({k: {m: float(np.mean(v)) for m, v in val.items()} 
               for k, val in cv_results.items()}, f, indent=2)

print('\n✓ Baseline evaluation complete!')